In [ ]:
!pip install -q datasets chromadb sentence-transformers langchain-text-splitters 

In [ ]:
# ==============================================================================
# 1. DEPENDENCIES & SETUP (Run this first if needed: !pip install -q datasets chromadb sentence-transformers langchain-text-splitters pandas)
# ==============================================================================
import os
import uuid
import pandas as pd
import numpy as np
import chromadb
from chromadb.utils import embedding_functions
from datasets import load_dataset
from sentence_transformers import CrossEncoder
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Kaggle environment setup
KAGGLE_WORK_DIR = "/kaggle/working"
DB_PATH = os.path.join(KAGGLE_WORK_DIR, "chromadb_qasper_eval")
os.makedirs(DB_PATH, exist_ok=True)


# ==============================================================================
# 2. RAG PIPELINE FUNCTIONS
# ==============================================================================
def chunk_academic_paper(text: str) -> list[str]:
    custom_separators = [
        "\nReferences\n",
        "\nAbstract\n",
        "\nIndex Terms:\n",
        "\nI\nIntroduction",
        "\nII\n",
        "\nIII\n",
        "\nIV\n",
        "\n\n",
        "\n",
        " ",
    ]
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=600,
        chunk_overlap=100,
        separators=custom_separators,
        is_separator_regex=False,
    )
    return text_splitter.split_text(text)


def embed_and_store(content: str, filename: str):
    client = chromadb.PersistentClient(path=DB_PATH)
    embedding_func = embedding_functions.SentenceTransformerEmbeddingFunction(
        model_name="all-MiniLM-L6-v2"
    )
    collection = client.get_or_create_collection(
        name="academic_papers",
        embedding_function=embedding_func,
        metadata={"hnsw:space": "cosine"},
    )

    existing_records = collection.get(where={"source": filename}, include=["metadatas"])
    if existing_records and len(existing_records["ids"]) > 0:
        return  # Skip if already ingested

    chunks = chunk_academic_paper(content)
    if not chunks:
        return

    chunk_ids = [f"{filename}_chunk_{i}" for i in range(len(chunks))]
    metadatas = [{"source": filename, "chunk_index": i} for i in range(len(chunks))]
    collection.upsert(documents=chunks, ids=chunk_ids, metadatas=metadatas)


def retrieve_top_k(query: str, k: int = 20) -> tuple[list[str], list[dict]]:
    client = chromadb.PersistentClient(path=DB_PATH)
    embedding_func = embedding_functions.SentenceTransformerEmbeddingFunction(
        model_name="all-MiniLM-L6-v2"
    )
    try:
        collection = client.get_collection(
            name="academic_papers", embedding_function=embedding_func
        )
    except Exception:
        return [], []

    results = collection.query(query_texts=[query], n_results=k)
    docs = results.get("documents", [[]])[0]
    metas = results.get("metadatas", [[]])[0]
    return docs, metas


def rerank_results(
    query: str,
    retrieved_chunks: list[str],
    retrieved_metadatas: list[dict],
    top_n: int = 5,
):
    if not retrieved_chunks:
        return []

    # Kaggle GPU utilization for CrossEncoder
    model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device="cuda")
    pairs = [[query, chunk] for chunk in retrieved_chunks]
    scores = model.predict(pairs)

    scored_items = sorted(
        zip(scores, retrieved_chunks, retrieved_metadatas),
        key=lambda x: x[0],
        reverse=True,
    )
    return scored_items[:top_n]


def search_local_memory(query: str) -> tuple[str, list[dict]]:
    candidates, metadatas = retrieve_top_k(query=query, k=20)
    top_items = rerank_results(
        query=query,
        retrieved_chunks=candidates,
        retrieved_metadatas=metadatas,
        top_n=10,
    )  # Using top 10 to measure Recall up to 10

    if not top_items:
        return "No relevant local literature found.", []

    context_chunks = []
    evidence_list = []

    for score, chunk, meta in top_items:
        context_chunks.append(f"Local Paper Fragment:\n{chunk}")
        evidence_list.append(
            {
                "source": meta.get("source", "Unknown Document"),
                "score": round(float(score), 4),
            }
        )

    context = "\n\n---\n\n".join(context_chunks)
    return context, evidence_list


# ==============================================================================
# 3. GUARDRAIL LOGIC
# ==============================================================================
def block_useless_memory_guardrail(context_output: str) -> bool:
    """
    Evaluates the output of search_local_memory.
    Returns False (Blocked) if results are useless, True (Valid) if good context is found.
    """
    if "No relevant local literature found" in context_output:
        return False
    # If it returns fragments but they are critically low quality, you could add threshold logic here
    return True


# ==============================================================================
# 4. DATASET PREPARATION & INGESTION
# ==============================================================================
print("⏳ Loading QASper Dataset (Subset of 100 papers for evaluation)...")
# We load 100 papers to ensure the evaluation completes in a reasonable time on Kaggle
dataset = load_dataset("allenai/qasper", split="train[:100]")

print(f"📥 Ingesting {len(dataset)} full-text papers into ChromaDB...")
for row in dataset:
    paper_id = row["id"]
    full_text_blocks = [f"Title: {row['title']}\nAbstract: {row['abstract']}"]

    # Flatten QASper nested full text
    for section_name, paragraphs in zip(
        row["full_text"]["section_name"], row["full_text"]["paragraphs"]
    ):
        full_text_blocks.append(f"\n{section_name}\n")
        full_text_blocks.extend(paragraphs)

    full_document_text = "\n".join(full_text_blocks)
    embed_and_store(full_document_text, filename=paper_id)
print("✅ Ingestion Complete!")

# Extract queries and ground truths
queries = []
for row in dataset:
    gt_paper_id = row["id"]
    for qa in row["qas"]["question"]:
        queries.append({"query": qa, "gt_id": gt_paper_id})

print(f"🎯 Prepared {len(queries)} evaluation questions.")


# ==============================================================================
# 5. EVALUATION LOOP (MRR & RECALL@K)
# ==============================================================================
def calculate_mrr(retrieved_ids: list[str], ground_truth_id: str) -> float:
    try:
        rank = retrieved_ids.index(ground_truth_id) + 1
        return 1.0 / rank
    except ValueError:
        return 0.0


def calculate_recall_at_k(
    retrieved_ids: list[str], ground_truth_id: str, k: int
) -> int:
    return 1 if ground_truth_id in retrieved_ids[:k] else 0


mrr_scores, recall_at_1, recall_at_5, recall_at_10 = [], [], [], []

print("🚀 Running Evaluation Pipeline...")
for i, item in enumerate(queries):
    query = item["query"]
    gt_id = item["gt_id"]

    # Run pipeline purely through your API-style function
    context, evidence = search_local_memory(query)

    # Trigger Guardrail to simulate agent checking
    is_valid_context = block_useless_memory_guardrail(context)

    # Deduplicate retrieved paper IDs while preserving rank order
    ranked_paper_ids = []
    if is_valid_context:
        for ev in evidence:
            pid = ev["source"]
            if pid not in ranked_paper_ids:
                ranked_paper_ids.append(pid)

    # Calculate & Store Metrics
    mrr_scores.append(calculate_mrr(ranked_paper_ids, gt_id))
    recall_at_1.append(calculate_recall_at_k(ranked_paper_ids, gt_id, 1))
    recall_at_5.append(calculate_recall_at_k(ranked_paper_ids, gt_id, 5))
    recall_at_10.append(calculate_recall_at_k(ranked_paper_ids, gt_id, 10))

    if (i + 1) % 25 == 0:
        print(f"   ... Processed {i + 1}/{len(queries)} queries")

# ==============================================================================
# 6. RESULTS SUMMARY
# ==============================================================================
results_summary = {
    "Metric": ["Mean Reciprocal Rank (MRR)", "Recall@1", "Recall@5", "Recall@10"],
    "Score": [
        np.mean(mrr_scores),
        np.mean(recall_at_1),
        np.mean(recall_at_5),
        np.mean(recall_at_10),
    ],
}

df_results = pd.DataFrame(results_summary)
print("\n" + "=" * 45)
print(" 📊 QASPER RAG PIPELINE EVALUATION SUMMARY")
print("=" * 45)
print(df_results.to_string(index=False))
print("=" * 45)